In [ ]:
#This version is a full variable load version

In [1]:
 
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# res='1km'
# Np_str='1e6'

# dx = 1 km; Np = 1M; Nt = 1 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_1min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_1min.nc') #***
res='1km'
Np_str='1e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [2]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [9]:
###########################################################################################################################################################################

In [10]:
#MAKING LAGRANGIAN BINARY ARRAY
###############################################################

In [18]:
#Lagrangian Position Arrays
##############
def grid_location(z,y,x):
    zf=data['zf'].values*1000; which_zh=np.clip(np.searchsorted(zf,z)-1,0,None)
    #which_zh=np.where(which_zh == -1, 0, which_zh) 
    
    yf=data['yf'].values*1000; which_yh=np.clip(np.searchsorted(yf,y)-1,0,None) 
    #which_yh=np.where(which_yh == -1, 0, which_yh) 
    
    xf=data['xf'].values*1000; which_xh=np.clip(np.searchsorted(xf,x)-1,0,None);
    #which_xh=np.where(which_xh == -1, 0, which_xh) 
    
    return which_zh,which_yh,which_xh

In [ ]:
print('Loading Lagrangian x,y,z into Memory\n')

dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# out_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5'
out_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_TEST.h5' 

# x=parcel['x'].data;y=parcel['y'].data;z=parcel['z'].data
with h5py.File(out_file, 'w') as f:
    x=parcel['x'].data;y=parcel['y'].data;z=parcel['z'].data
    Z,Y,X=np.zeros_like(z),np.zeros_like(y),np.zeros_like(x)

    
    # Create datasets for z, y, and x
    f.create_dataset('z', data=z, dtype='float32')  # Store z as float32 (f4)
    f.create_dataset('y', data=y, dtype='float32')  # Store y as float32 (f4)
    f.create_dataset('x', data=x, dtype='float32')

    f.create_dataset('Z', data=np.zeros_like(z), dtype='int')
    f.create_dataset('Y', data=np.zeros_like(y), dtype='int')
    f.create_dataset('X', data=np.zeros_like(x), dtype='int')

del z,y,x,Z,Y,X

In [ ]:
print('Getting Z,Y,X\n')
with h5py.File(out_file, 'r+') as f: #'w': write, 'r': read, 'r+': read-write

    # Read the data from the file
    z = f['z'][:]
    y = f['y'][:]
    x = f['x'][:]

    # Calculate Z, Y, X using the grid_location function
    Z_temp, Y_temp, X_temp = grid_location(z, y, x)

    f['Z'][:] = Z_temp
    f['Y'][:] = Y_temp
    f['X'][:] = X_temp

In [4]:
def call_variable(varname):
    var_data=data[varname].data
    return var_data

In [5]:
def make_lagrangian_array(varname):
    var_data=call_variable(varname)
    VAR=np.zeros(parcel['z'].shape,dtype='float32')

    with h5py.File(out_file, 'r'):
        
        Nt=len(data['time'])
        Np=len(parcel['xh'])
        for p in np.arange(Np):
            if np.mod(p,2e5)==0: print(f"{p}/{len(parcel['xh'])}")
        
            #Get Indicies
            zs=f['Z'][:,p]
            ys=f['Y'][:,p]
            xs=f['X'][:,p]
            ts = np.arange(Nt)  
        
            #Get Values
            vars = var_data[ts, zs, ys, xs]
    
            #Adding to Lagrangian Array
            VAR[:,p]=vars
    
            del vars
    del var_data
    return VAR

In [6]:
import h5py

print('Making W, QC, and QI Lagrangian Array\n')

# Open the h5py file once and write variables
with h5py.File(out_file, 'r+') as f:
    
    # Creating arrays and checking memory
    W = make_lagrangian_array('winterp'); # check_memory() 
    QC = make_lagrangian_array('qc'); # check_memory()
    QI = make_lagrangian_array('qi'); # check_memory()
    
    print('Making QC+QI Lagrangian Array\n')
    import dask.array as da
    Nt = len(data['time'])
    QC = da.from_array(QC, chunks=(Nt, 'auto'))
    QI = da.from_array(QI, chunks=(Nt, 'auto'))
    QCQI = QC + QI
    QCQI = QCQI.compute()
    check_memory()

    array_to_dask = True

    # Add the variables W and QCQI to the h5py file, if they don't already exist
    if 'W' not in f:
        f.create_dataset('W', data=W)
    if 'QCQI' not in f:
        f.create_dataset('QCQI', data=QCQI)


0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'NamespaceMagics': '0.0 MB', 'open': '0.0 MB', 'coefs': '0.0 MB'}

3.724 GB in use overall
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QC': '532.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'NamespaceMagics': '0.0 MB', 'open': '0.0 MB'}

4.256 GB in use overall
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QC': '532.0 MB', 'QI': '532.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'Na

In [7]:
#Create Set Thresholds and Create Binary Arrays
w_thresh1=0.1
w_thresh2=0.5

qcqi_thresh1=1e-6
qcqi_thresh2=1e-6



print('Making Lagrangian Binary Arrays\n')

with h5py.File(out_file, 'r+') as f:
    #Apply Thresholds 
    if array_to_dask==True:
        import dask.array as da
        Nt=len(data['time'])
        W = da.from_array(W, chunks=(Nt, 'auto'))
        QCQI = da.from_array(QCQI, chunks=(Nt, 'auto'))
        array_to_dask=False

    # Apply thresholds on Dask arrays (this will be done lazily)
    print('Making A_g')
    A_g = da.where((W >= w_thresh1) & (QCQI < qcqi_thresh1), 1, 0)
    A_g = A_g.compute()

    print('Making A_c')
    A_c = da.where((W >= w_thresh2) & (QCQI >= qcqi_thresh2), 1, 0)
    A_c = A_c.compute()    
    
    # Saving Data
    ##############
    print('Saving Data\n')
    if 'A_g' not in f:
        f.create_dataset('A_g', data=A_g)
    if 'A_c' not in f:
        f.create_dataset('A_c', data=A_c)

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'A_g': '1064.0 MB', 'A_c': '1064.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'NamespaceMagics': '0.0 MB', 'open': '0.0 MB'}

5.32 GB in use overall


In [6]:
#########################################

In [8]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min.h5'
with h5py.File(in_file, 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    # A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    # Z = f['Z'][:]
    # Y = f['Y'][:]
    # X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [21]:
# #TESTING CHECKING THAT THRESHOLDS WORKED
# # w_data=data['winterp'].data #interpolation w data z coordinate from zh to zf
# # variable='qc'; qc_data=data[variable].data # get qc data
# # variable='qi'; qi_data=data[variable].data # get qc data
# # qc_plus_qi=qc_data+qi_data

# def test_thresholds(t,type):
#     # w_data=data['w'].interp(zf=data['zh']).data

#     if type=='g':
#         A=A_g
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'max qcqi is {hey.max()}')
#     if type=='c':
#         A=A_c
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min qcqi is {hey.min()}')

# #GENERAL UPDRAFT
# t=70
# test_thresholds(t,'g')
# print('\n')
# #CLOUDY UPDRAFT
# test_thresholds(t,'c')

min w is 0.10000452399253845
max qcqi is 9.999595595999722e-10


min w is 0.5002251863479614
min qcqi is 1.096519213206193e-06
